# 🏫 Caso de Estudio: Calidad Educativa y Factores Socioeconómicos en Perú
**Especialización en Business Intelligence & Business Analytics**  
**Módulo: Calidad de Datos y Analítica Avanzada**

---

## 📋 Pregunta Analítica Central
> *¿Qué combinación de factores operacionales de la institución educativa y socioeconómicos del distrito predice mejor la probabilidad de que una IE tenga un nivel de logro satisfactorio bajo y un clima escolar desfavorable, y qué segmentos de instituciones emergen de ese patrón?*

---
### Fuentes de datos utilizadas
| # | Dataset | Fuente | Formato |
|---|---------|--------|---------|
| D1 | Resultados ECE | datos.gob.pe / MINEDU | CSV |
| D2 | Padrón de IE | datos.gob.pe / MINEDU | CSV |
| D3 | Indicadores World Bank | databank.worldbank.org | API REST / JSON |
| D4 | Índice Vulnerabilidad Distrital | Kaggle / INEI | Excel (.xlsx) |


## ⚙️ Configuración del Entorno

In [ ]:
# ── Instalación de librerías necesarias ──────────────────────────────────────
!pip install -q missingno scikit-learn xgboost openpyxl unidecode

# ── Importaciones ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import requests
import warnings
import re
from io import StringIO
from unidecode import unidecode
from scipy import stats
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import MinMaxScaler, RobustScaler, LabelEncoder
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Paleta de colores del proyecto
VERDE = '#2D6A4F'
VERDE_CLARO = '#52B788'
AMARILLO = '#F4A261'
ROJO = '#E63946'
GRIS = '#6C757D'

print('✅ Entorno configurado correctamente.')

---
## PARTE 1 — Ingesta de Datos
> **Objetivo:** Cargar los cuatro datasets detectando y resolviendo problemas de formato antes de cualquier análisis.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET 1 — Resultados ECE (Datos Abiertos MINEDU)
# URL real: https://www.datosabiertos.gob.pe/dataset/resultados-ece
# ═══════════════════════════════════════════════════════════════════════════════

URL_D1 = 'https://raw.githubusercontent.com/datasets/minedu-ece-peru/main/ece_resultados_2023.csv'

# PROBLEMA DOCUMENTADO: encoding utf-8-sig (BOM invisible) y dtype=str para
# preservar ceros iniciales en Código Modular.
try:
    d1 = pd.read_csv(URL_D1, encoding='utf-8-sig', dtype={'cod_mod': str, 'ubigeo': str})
    print(f'✅ D1 cargado desde GitHub: {d1.shape}')
except Exception as e:
    print(f'⚠️  Descarga directa fallida ({e}). Generando dataset simulado...')

    # ── Dataset simulado fiel al esquema real ECE ────────────────────────────
    np.random.seed(42)
    N = 8000
    departamentos = ['LIMA', 'AREQUIPA', 'CUSCO', 'PIURA', 'LA LIBERTAD',
                     'JUNIN', 'PUNO', 'CAJAMARCA', 'AYACUCHO', 'LORETO']
    ugels = [f'UGEL-{i:02d}' for i in range(1, 31)]
    grados_areas = ['2do_Prim_Lectura', '2do_Prim_Matematica', '4to_Prim_Lectura', '4to_Prim_Matematica']

    dept_col = np.random.choice(departamentos, N)
    area_col = np.random.choice(['Urbano', 'Rural'], N, p=[0.6, 0.4])
    gestion_col = np.random.choice(['Publica', 'Privada'], N, p=[0.75, 0.25])

    # Logro satisfactorio depende de área y gestión (realismo)
    base_logro = np.where(area_col=='Urbano', 40, 18) + np.where(gestion_col=='Privada', 15, 0)
    logro_sat = np.clip(np.random.normal(base_logro, 12), 0, 100).round(1)

    # Introducir NaN reales (~22%) y strings inválidos
    mask_nan = np.random.random(N) < 0.22
    logro_str = np.where(mask_nan,
                         np.random.choice(['Sin Información', 'S/D', ''], N),
                         logro_sat.astype(str))

    d1 = pd.DataFrame({
        'cod_mod': [f'{i:07d}' for i in np.random.randint(1000000, 9999999, N)],
        'nombre_ie': [f'I.E. {np.random.choice(["SAN MARTIN","JOSE CARLOS MARIATEGUI","REPUBLICA DEL PERU","SANTA ROSA","LOS ANDES"])} N°{np.random.randint(100,999)}' for _ in range(N)],
        'dre': dept_col,
        'ugel': np.random.choice(ugels, N),
        'distrito': [f'DISTRITO_{np.random.randint(1, 200):03d}' for _ in range(N)],
        'area_geografica': area_col,
        'gestion_tipo': gestion_col,
        'grado_area': np.random.choice(grados_areas, N),
        'pct_logro_previo_inicio': np.random.uniform(5, 40, N).round(1),
        'pct_logro_inicio': np.random.uniform(15, 45, N).round(1),
        'pct_logro_proceso': np.random.uniform(20, 40, N).round(1),
        'pct_logro_satisfactorio': logro_str,  # ← STRING con NaN disfrazados
        'anio_evaluacion': 2023,
    })

    # Duplicados intencionales (IE fusionadas no depuradas)
    duplicados = d1.sample(50).copy()
    duplicados['cod_mod'] = [f'{i:07d}' for i in np.random.randint(1000000, 9999999, 50)]
    d1 = pd.concat([d1, duplicados], ignore_index=True)

    print(f'✅ Dataset D1 simulado generado: {d1.shape}')

d1.head(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET 2 — Padrón de Instituciones Educativas (ESCALE / Datos Abiertos Perú)
# URL real: https://www.datosabiertos.gob.pe/dataset/padron-de-instituciones-educativas
# ═══════════════════════════════════════════════════════════════════════════════

try:
    URL_D2 = 'https://raw.githubusercontent.com/datasets/minedu-padron/main/padron_ie_2023.csv'
    d2 = pd.read_csv(URL_D2, encoding='utf-8-sig', dtype={'cod_mod': str, 'ubigeo': str})
    print(f'✅ D2 cargado: {d2.shape}')
except:
    print('⚠️  Generando Dataset 2 (Padrón IE) simulado...')
    np.random.seed(7)
    cod_mods_unicos = list(d1['cod_mod'].unique()[:5500])
    M = len(cod_mods_unicos)

    forma = np.random.choice(['Polidocente', 'Multigrado', 'Unidocente', None],
                              M, p=[0.55, 0.25, 0.12, 0.08])
    n_doc = np.random.randint(1, 60, M).astype(float)
    n_doc[np.random.random(M) < 0.05] = 0       # ceros sospechosos
    n_doc[np.random.random(M) < 0.04] = np.nan  # faltantes

    n_est = np.random.randint(10, 800, M).astype(float)
    n_est[np.random.random(M) < 0.03] = np.nan

    internet_vals = np.random.choice(['SI', 'NO', 'Sin dato'], M, p=[0.35, 0.50, 0.15])
    internet_vals[np.random.random(M) < 0.05] = np.nan

    lat = np.random.uniform(-18, -3, M)
    lon = np.random.uniform(-81, -69, M)
    lat[np.random.random(M) < 0.12] = np.nan
    lon[np.random.random(M) < 0.12] = np.nan

    tipo_eib = np.full(M, np.nan, dtype=object)
    eib_idx = np.random.choice(M, int(M*0.4), replace=False)
    tipo_eib[eib_idx] = np.random.choice(['EIB_FORTALECIMIENTO', 'EIB_REVITALIZACION', None], len(eib_idx), p=[0.2, 0.15, 0.65])

    d2 = pd.DataFrame({
        'cod_mod': cod_mods_unicos,
        'nivel': np.random.choice(['Primaria', 'Secundaria', 'Inicial'], M, p=[0.5, 0.35, 0.15]),
        'modalidad': np.random.choice(['Basica Regular', 'Basica Alternativa', 'Especial'], M, p=[0.92, 0.05, 0.03]),
        'forma': forma,
        'turno': np.random.choice(['Manana', 'Tarde', 'Completo'], M, p=[0.6, 0.25, 0.15]),
        'n_docentes': n_doc,
        'n_estudiantes': n_est,
        'conectividad_internet': internet_vals,
        'tipo_eib': tipo_eib,
        'latitud': lat,
        'longitud': lon,
        'dre': np.random.choice(d1['dre'].unique(), M),
        'distrito': np.random.choice(d1['distrito'].unique(), M),
    })

    print(f'✅ Dataset D2 simulado generado: {d2.shape}')

d2.head(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET 3 — World Bank API (Indicadores para Perú)
# ═══════════════════════════════════════════════════════════════════════════════

def get_wb_indicator(indicator_code, country='PE', years=range(2018, 2024)):
    """Consume la API REST del World Bank con manejo de paginación."""
    base_url = 'https://api.worldbank.org/v2/country/{}/indicator/{}'.format(country, indicator_code)
    params = {'format': 'json', 'per_page': 100, 'mrv': 6}
    records = []
    try:
        resp = requests.get(base_url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            # PROBLEMA: la API devuelve lista de 2 elementos — el segundo son los datos
            if len(data) == 2 and data[1]:
                for item in data[1]:
                    records.append({
                        'anio': int(item.get('date', 0)),
                        'valor': item.get('value'),
                        'indicador': indicator_code,
                    })
    except Exception as e:
        print(f'  ⚠️ Error API {indicator_code}: {e}')
    return pd.DataFrame(records)

indicadores = {
    'SE.XPD.TOTL.GD.ZS': 'gasto_educacion_pct_pib',
    'SE.PRM.ENRL.TC.ZS':  'ratio_estudiante_docente_primaria',
    'SE.ADT.LITR.ZS':     'tasa_alfabetizacion_adulta',
    'EG.ELC.ACCS.RU.ZS':  'acceso_electricidad_rural_pct',
}

print('Consultando World Bank API...')
frames = []
for code, nombre in indicadores.items():
    df_ind = get_wb_indicator(code)
    if not df_ind.empty:
        df_ind['nombre_indicador'] = nombre
        frames.append(df_ind)
        print(f'  ✅ {nombre}: {len(df_ind)} registros')
    else:
        print(f'  ⚠️ {nombre}: sin datos (generando simulado)')
        # Generar datos simulados coherentes con Perú
        vals = {'SE.XPD.TOTL.GD.ZS': [3.8,3.9,3.7,3.6,3.8,3.9],
                'SE.PRM.ENRL.TC.ZS':  [17.2,16.8,16.5,15.9,16.1,15.7],
                'SE.ADT.LITR.ZS':     [94.1,94.3,94.5,94.6,94.8,95.0],
                'EG.ELC.ACCS.RU.ZS':  [78.0,80.5,82.3,84.1,85.7,87.2]}.get(code, [None]*6)
        df_ind = pd.DataFrame({'anio': list(range(2018,2024)), 'valor': vals,
                               'indicador': code, 'nombre_indicador': nombre})
        frames.append(df_ind)

d3_largo = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# Pivotar a formato ancho: una fila por año
d3 = d3_largo.pivot_table(index='anio', columns='nombre_indicador', values='valor').reset_index()
d3.columns.name = None
# Usar el año más reciente disponible para el JOIN
d3_2023 = d3[d3['anio'] == d3['anio'].max()].copy()
print(f'\n✅ D3 (World Bank) listo: {d3_2023.shape}')
d3_2023

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET 4 — Índice de Vulnerabilidad Distrital (Kaggle / INEI)
# URL: https://www.kaggle.com/datasets/peruinei/pobreza-distrital-peru
# ═══════════════════════════════════════════════════════════════════════════════

# OPCIÓN A: Si ya descargaste el archivo de Kaggle, cárgalo así:
# from google.colab import files
# uploaded = files.upload()  # subir el .xlsx
# d4 = pd.read_excel('vulnerabilidad_distrital.xlsx', sheet_name='Datos',
#                    dtype={'ubigeo': str})

# OPCIÓN B (demo): Dataset simulado con la misma estructura y problemas reales
print('⚠️  Generando Dataset 4 (Vulnerabilidad Distrital) simulado...')
print('   Para usar datos reales: descarga de Kaggle y descomenta OPCIÓN A arriba.')
np.random.seed(99)

departamentos_peru = ['LIMA', 'AREQUIPA', 'CUSCO', 'PIURA', 'LA LIBERTAD',
                      'JUNIN', 'PUNO', 'CAJAMARCA', 'AYACUCHO', 'LORETO']
distritos_por_dept = 20
K = len(departamentos_peru) * distritos_por_dept

dept_d4 = np.repeat(departamentos_peru, distritos_por_dept)
dist_d4 = [f'DISTRITO_{i:03d}' for i in range(1, K+1)]
ubigeo_d4 = [f'{(idx//distritos_por_dept+1):02d}{(idx%distritos_por_dept+1):03d}01' for idx in range(K)]

# Pobreza más alta en regiones menos desarrolladas (realismo)
pobreza_base = {'LIMA':8, 'AREQUIPA':12, 'CUSCO':28, 'PIURA':25, 'LA LIBERTAD':22,
                'JUNIN':24, 'PUNO':35, 'CAJAMARCA':40, 'AYACUCHO':38, 'LORETO':42}
pobreza = np.array([np.clip(np.random.normal(pobreza_base[d], 8), 2, 75) for d in dept_d4])

# PROBLEMA REAL: coma decimal en lugar de punto en algunos valores
pobreza_str = [f'{v:.1f}'.replace('.', ',') if np.random.random() < 0.08 else f'{v:.1f}'
               for v in pobreza]

idh = np.clip(np.random.normal(0.62, 0.08, K), 0.35, 0.85).round(3)
idh[np.random.random(K) < 0.05] = np.nan

sin_internet = np.clip(100 - np.random.normal(35, 20, K), 20, 95).round(1)
sin_internet[np.random.random(K) < 0.07] = np.nan

d4 = pd.DataFrame({
    'ubigeo': ubigeo_d4,
    'departamento': dept_d4,
    'distrito': dist_d4,
    'pct_pobreza_monetaria': pobreza_str,  # ← STRING con coma decimal (bug real)
    'pct_pobreza_extrema': np.clip(pobreza * 0.4 + np.random.normal(0, 3, K), 0, 40).round(1),
    'idh': idh,
    'pct_hogares_sin_internet': sin_internet,
    'tasa_desempleo': np.clip(np.random.normal(5.5, 2, K), 1, 18).round(1),
    'pct_poblacion_rural': np.clip(np.random.normal(45, 25, K), 0, 100).round(1),
})

print(f'✅ Dataset D4 simulado generado: {d4.shape}')
d4.head(3)

---
## 📁 Creación de Estructura de Directorios
> **Objetivo:** Organizar el espacio de trabajo creando los directorios base para datos originales, procesamiento y resultados.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ESTRUCTURA FÍSICA DEL ESPACIO DE TRABAJO
# Se crean los directorios raíz y sus tres subdivisiones
# ═══════════════════════════════════════════════════════════════════════════════
from pathlib import Path

# ── Definición de rutas ───────────────────────────────────────────────────────
Directorio_Base       = Path('DAA_Grupo06')
Directorio_Datos      = Directorio_Base / 'Data_Original'
Directorio_Procesos   = Directorio_Base / 'Data_Procesamiento'
Directorio_Resultados = Directorio_Base / 'Data_Resultados'

# ── Creación de directorios (no falla si ya existen) ─────────────────────────
for d in [Directorio_Datos, Directorio_Procesos, Directorio_Resultados]:
    d.mkdir(parents=True, exist_ok=True)

# ── Guardar datasets cargados en Data_Original ───────────────────────────────
d1.to_csv(Directorio_Datos / 'D1_ECE_resultados.csv',   index=False, encoding='utf-8-sig')
d2.to_csv(Directorio_Datos / 'D2_Padron_IE.csv',        index=False, encoding='utf-8-sig')
d3.to_csv(Directorio_Datos / 'D3_WorldBank.csv',        index=False, encoding='utf-8-sig')
d4.to_csv(Directorio_Datos / 'D4_Vulnerabilidad.csv',   index=False, encoding='utf-8-sig')

# ── Reporte de estructura creada ─────────────────────────────────────────────
print('✅ Estructura de directorios creada:')
print(f'   📂 {Directorio_Base}/')
print(f'   ├── 📁 {Directorio_Datos.name}/         ← datasets originales (D1–D4)')
print(f'   ├── 📁 {Directorio_Procesos.name}/  ← datos en proceso de limpieza')
print(f'   └── 📁 {Directorio_Resultados.name}/      ← outputs finales y modelos')
print()
print('✅ Archivos guardados en Data_Original:')
for f in sorted(Directorio_Datos.iterdir()):
    print(f'   • {f.name}')

---
## PARTE 2 — Perfilado de Datos
> **Objetivo:** Entender la estructura, cardinalidad y calidad superficial de cada dataset antes de modificarlo.

In [ ]:
def perfilar_dataset(df, nombre):
    """Genera un perfil completo de un DataFrame."""
    print(f'\n{'='*60}')
    print(f'  PERFIL: {nombre}')
    print(f'{'='*60}')
    print(f'  Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
    print(f'  Filas completamente duplicadas: {df.duplicated().sum()}')

    perfil = pd.DataFrame({
        'Tipo': df.dtypes,
        'Nulos': df.isnull().sum(),
        'Pct_Nulos': (df.isnull().sum() / len(df) * 100).round(2),
        'Cardinalidad': df.nunique(),
        'Muestra': [str(df[c].dropna().iloc[0]) if df[c].dropna().shape[0] > 0 else 'N/A'
                    for c in df.columns]
    })
    perfil = perfil.sort_values('Pct_Nulos', ascending=False)
    display(perfil.style.background_gradient(subset=['Pct_Nulos'], cmap='Oranges'))
    return perfil

for nombre, df in [('D1 – Resultados ECE', d1),
                   ('D2 – Padrón IE', d2),
                   ('D3 – World Bank', d3),
                   ('D4 – Vulnerabilidad Distrital', d4)]:
    perfilar_dataset(df, nombre)

In [ ]:
# Verificar unicidad de claves primarias
print('── Verificación de unicidad de claves ──────────────────────')
for nombre, df, clave in [
    ('D2 – Padrón IE', d2, 'cod_mod'),
    ('D4 – Vulnerabilidad', d4, 'ubigeo'),
]:
    duplicados = df[clave].duplicated().sum()
    estado = '✅' if duplicados == 0 else '⚠️ DUPLICADOS'
    print(f'  {estado} {nombre} — {clave}: {duplicados} duplicados')

# Validar dominios
print('\n── Validación de dominios ──────────────────────────────────')
print(f'  D1 area_geografica valores únicos: {d1["area_geografica"].unique()}')
print(f'  D1 gestion_tipo valores únicos: {d1["gestion_tipo"].unique()}')
print(f'  D2 conectividad_internet valores únicos: {d2["conectividad_internet"].unique()}')

---
## PARTE 3 — Calidad de Datos
> **Objetivo:** Evaluar las cinco dimensiones de calidad y documentar decisiones de tratamiento.

In [ ]:
print('═══ DIMENSIÓN 1: COMPLETITUD ═══')

# D1: % de IE con logro satisfactorio reportado
# PROBLEMA: la columna viene como string con valores 'Sin Información', 'S/D', ''
valores_invalidos = ['Sin Información', 'S/D', '', 'sin informacion', 'SIN INFORMACIÓN']
d1['pct_logro_satisfactorio_raw'] = d1['pct_logro_satisfactorio'].copy()
d1['pct_logro_satisfactorio'] = d1['pct_logro_satisfactorio'].replace(valores_invalidos, np.nan)
d1['pct_logro_satisfactorio'] = pd.to_numeric(d1['pct_logro_satisfactorio'], errors='coerce')

completitud_d1 = (d1['pct_logro_satisfactorio'].notna().sum() / len(d1) * 100)
print(f'  D1 — IE con % Satisfactorio reportado: {completitud_d1:.1f}%')
print(f'  D1 — Strings inválidos encontrados: {(d1["pct_logro_satisfactorio_raw"].isin(valores_invalidos)).sum()}')

# Columnas con > 30% nulos en D2
nulos_d2 = (d2.isnull().sum() / len(d2) * 100).sort_values(ascending=False)
print(f'\n  D2 — Variables con >30% nulos:')
print(nulos_d2[nulos_d2 > 30].to_string())

print('\n═══ DIMENSIÓN 2: EXACTITUD ═══')
# Porcentajes fuera de rango [0, 100]
invalid_logro = d1[(d1['pct_logro_satisfactorio'] < 0) | (d1['pct_logro_satisfactorio'] > 100)]
print(f'  D1 — % Satisfactorio fuera de rango [0,100]: {len(invalid_logro)} registros')

# PROBLEMA: % Pobreza con coma decimal en D4
d4['pct_pobreza_str_backup'] = d4['pct_pobreza_monetaria'].copy()
d4['pct_pobreza_monetaria'] = d4['pct_pobreza_monetaria'].astype(str).str.replace(',', '.').pipe(pd.to_numeric, errors='coerce')
comas_corregidas = (d4['pct_pobreza_str_backup'].str.contains(',', na=False)).sum()
print(f'  D4 — Valores con coma decimal corregidos: {comas_corregidas}')

print('\n═══ DIMENSIÓN 3: CONSISTENCIA ═══')
if 'anio_evaluacion' in d1.columns:
    print(f'  D1 — Años de evaluación únicos: {sorted(d1["anio_evaluacion"].unique())}')

print('\n═══ DIMENSIÓN 4: UNICIDAD ═══')
dup_d1_nombres = d1.duplicated(subset=['nombre_ie', 'grado_area']).sum()
print(f'  D1 — IE con mismo nombre y grado duplicados: {dup_d1_nombres}')

print('\n═══ DIMENSIÓN 5: VALIDEZ ═══')
dres_validas = ['LIMA', 'AREQUIPA', 'CUSCO', 'PIURA', 'LA LIBERTAD',
                'JUNIN', 'PUNO', 'CAJAMARCA', 'AYACUCHO', 'LORETO']
invalidas = d1[~d1['dre'].isin(dres_validas)]['dre'].nunique()
print(f'  D1 — DRE con nombres fuera del dominio esperado: {invalidas}')

---
## PARTE 4 — Diagnóstico y Tratamiento de Valores Faltantes
> **Proceso de 9 pasos con 4 tests estadísticos y comparación de 4 métodos de imputación.**

In [ ]:
# ── Preparar dataset para análisis de missings ────────────────────────────────
# Pivotar D1 a formato ancho (una fila por IE, métricas por grado/área como columnas)
d1_ancho = d1.pivot_table(
    index=['cod_mod', 'dre', 'area_geografica', 'gestion_tipo', 'distrito'],
    columns='grado_area',
    values='pct_logro_satisfactorio',
    aggfunc='mean'
).reset_index()
d1_ancho.columns.name = None

# JOIN D1 + D2
master = pd.merge(d1_ancho, d2, on='cod_mod', how='left', suffixes=('_ece', '_padron'))
print(f'Tabla maestra tras JOIN D1+D2: {master.shape}')

# Normalizar columnas de distrito para JOIN con D4
def normalizar_texto(s):
    if pd.isna(s): return np.nan
    return unidecode(str(s)).upper().strip().replace('  ', ' ')

master['distrito_norm'] = master['distrito_ece' if 'distrito_ece' in master.columns else 'distrito'].apply(normalizar_texto)
master['dre_norm'] = master['dre'].apply(normalizar_texto)
d4['departamento_norm'] = d4['departamento'].apply(normalizar_texto)
d4['distrito_norm'] = d4['distrito'].apply(normalizar_texto)

# JOIN con D4
master = pd.merge(master, d4, left_on=['dre_norm', 'distrito_norm'],
                  right_on=['departamento_norm', 'distrito_norm'], how='left')

# Enriquecer con D3 (nivel departamental)
for col in d3_2023.columns:
    if col != 'anio':
        master[col] = d3_2023[col].values[0] if len(d3_2023) > 0 else np.nan

print(f'Tabla maestra final: {master.shape}')
master.head(2)

In [ ]:
# ── PASO 1: Tabla resumen de nulos ───────────────────────────────────────────
print('PASO 1 — Tabla resumen de valores faltantes')

cols_analisis = [c for c in master.columns
                 if master[c].dtype in ['float64', 'int64']
                 or c in ['conectividad_internet', 'forma', 'tipo_eib']]

resumen_nulos = pd.DataFrame({
    'Variable': cols_analisis,
    'N_Missing': [master[c].isnull().sum() for c in cols_analisis],
    'Pct_Missing': [(master[c].isnull().sum()/len(master)*100).round(2) for c in cols_analisis],
    'N_Completos': [master[c].notna().sum() for c in cols_analisis],
}).sort_values('Pct_Missing', ascending=False)

display(resumen_nulos.head(15).style.background_gradient(subset=['Pct_Missing'], cmap='Reds'))

In [ ]:
# ── PASO 2: Visualización de patrones con missingno ───────────────────────────
print('PASO 2 — Visualización de patrones de valores faltantes')

cols_num = master.select_dtypes(include='number').columns.tolist()[:15]
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

msno.matrix(master[cols_num], ax=axes[0], color=(0.18, 0.42, 0.31), sparkline=False)
axes[0].set_title('Matriz de Patrones de Nulos', fontsize=13, fontweight='bold', color='#1B4332')

msno.heatmap(master[cols_num], ax=axes[1], cmap='RdYlGn')
axes[1].set_title('Correlación entre Indicadores de Nulos', fontsize=13, fontweight='bold', color='#1B4332')

plt.tight_layout()
plt.savefig('paso2_missingno.png', dpi=150, bbox_inches='tight')
plt.show()
print('  💡 Interpretación: columnas con nulos alineados → patrón sistemático (MAR/MNAR), no aleatorio (MCAR)')

In [ ]:
# ── PASO 3: Test 1 — Aproximación a Little's MCAR ────────────────────────────
print('PASO 3 — Test 1: Aproximación Little MCAR (t-test de Welch)')
print('Hipótesis: si el dato falta completamente al azar (MCAR), las medias de otras variables')
print('deben ser iguales entre el grupo con datos y el grupo sin datos.\n')

var_objetivo = [c for c in cols_num if master[c].isnull().sum() > 0][:6]
vars_predictoras = [c for c in cols_num if master[c].isnull().sum() < len(master)*0.5][:8]

resultados_mcar = []
for var in var_objetivo:
    grupo_presente = master[master[var].notna()]
    grupo_ausente  = master[master[var].isna()]
    sig_count = 0
    for pred in vars_predictoras:
        if pred == var: continue
        a = grupo_presente[pred].dropna()
        b = grupo_ausente[pred].dropna()
        if len(a) > 5 and len(b) > 5:
            _, p = stats.ttest_ind(a, b, equal_var=False)
            if p < 0.05: sig_count += 1
    pct_sig = sig_count / len(vars_predictoras) * 100
    mcar_verdict = '✅ Compatible MCAR' if pct_sig <= 20 else '❌ NO MCAR (MAR/MNAR)'
    resultados_mcar.append({'Variable': var, 'Pct_Comparaciones_Significativas': round(pct_sig,1),
                             'Diagnóstico': mcar_verdict})

df_mcar = pd.DataFrame(resultados_mcar)
display(df_mcar)

In [ ]:
# ── PASO 4: Test 2 — Análisis de Patrones ────────────────────────────────────
print('PASO 4 — Test 2: Análisis de Patrones de Missing')

patrones = master[cols_num].isnull().value_counts()
n_patrones = len(patrones)
print(f'  Número de combinaciones únicas de nulos: {n_patrones}')
print(f'  Interpretación: {"Alta diversidad → MAR/MNAR" if n_patrones > 10 else "Pocos patrones → compatible con MCAR"}')
print('\n  Top 5 patrones más frecuentes:')
display(patrones.head())

In [ ]:
# ── PASO 5: Test 3 — Correlación entre Indicadores de Missing ────────────────
print('PASO 5 — Test 3: Correlación entre Indicadores de Missing')

miss_indicators = master[cols_num].isnull().astype(int)
# Solo columnas con al menos 2% de nulos
cols_con_nulos = [c for c in cols_num if master[c].isnull().sum() > len(master)*0.02]

if len(cols_con_nulos) >= 2:
    corr_miss = miss_indicators[cols_con_nulos].corr()
    fig, ax = plt.subplots(figsize=(10, 6))
    mask = np.triu(np.ones_like(corr_miss, dtype=bool))
    sns.heatmap(corr_miss, mask=mask, annot=True, fmt='.2f',
                cmap='RdYlGn_r', center=0, ax=ax,
                cbar_kws={'label': 'Correlación de Indicadores de Nulos'})
    ax.set_title('Test 3: Correlación entre Indicadores de Missing\n|r| > 0.3 → ausencias sistemáticas → MAR/MNAR',
                 fontsize=11, color='#1B4332', pad=10)
    plt.tight_layout()
    plt.savefig('paso5_corr_missing.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('  Insuficientes columnas con nulos para análisis de correlación.')

In [ ]:
# ── PASO 6: Test 4 — Kolmogorov-Smirnov ─────────────────────────────────────
print('PASO 6 — Test 4: Kolmogorov-Smirnov')
print('Compara la distribución de X entre el grupo donde Y tiene nulos y donde no.\n')

resultados_ks = []
for var_x in cols_num[:5]:
    for var_y in cols_con_nulos[:3]:
        if var_x == var_y: continue
        grupo_y_presente = master[master[var_y].notna()][var_x].dropna()
        grupo_y_ausente  = master[master[var_y].isna()][var_x].dropna()
        if len(grupo_y_presente) > 10 and len(grupo_y_ausente) > 10:
            ks_stat, p_val = stats.ks_2samp(grupo_y_presente, grupo_y_ausente)
            resultados_ks.append({
                'Var_X': var_x, 'Var_Y_con_nulos': var_y,
                'KS_stat': round(ks_stat, 3), 'p_valor': round(p_val, 4),
                'Diagnóstico': 'MAR/MNAR' if p_val < 0.05 else 'Compatible MCAR'
            })

df_ks = pd.DataFrame(resultados_ks)
if not df_ks.empty:
    display(df_ks.sort_values('p_valor').head(10))
else:
    print('  No hay suficientes combinaciones para el test KS.')

In [ ]:
# ── PASO 7: Diagnóstico Integrado ────────────────────────────────────────────
print('PASO 7 — Diagnóstico Integrado de Mecanismos de Ausencia')
print()

diagnostico = [
    ('pct_logro_satisfactorio',   'MAR',  'IE pequeñas no alcanzan N mínimo de estudiantes evaluados.',         'KNN Imputer usando n_estudiantes como predictor'),
    ('idh',                       'MAR',  'Distritos muy pequeños o recién creados sin cálculo IDH.',           'Imputación por mediana departamental'),
    ('pct_hogares_sin_internet',   'MAR',  'Distritos rurales remotos con mayor proporción de nulos.',           'KNN Imputer usando pct_pobreza y pct_rural como predictores'),
    ('tipo_eib',                   'MNAR', 'Solo aplica a IE EIB — ausencia define el grupo (no EIB).',          'NO IMPUTAR — crear indicador binario eib_flag'),
    ('latitud',                    'MAR',  'IE rurales remotas con menor probabilidad de tener coordenadas.',    'Imputar con centroide del distrito'),
    ('n_docentes',                 'MNAR', 'Posible sub-reporte de IE con problemas de gestión.',                'Winsorización + documentar como limitación'),
]

df_diag = pd.DataFrame(diagnostico, columns=['Variable', 'Mecanismo', 'Justificación', 'Estrategia'])
display(df_diag.style.applymap(
    lambda v: 'background-color: #D8F3DC' if v=='MAR' else ('background-color: #F8D7DA' if v=='MNAR' else ''),
    subset=['Mecanismo']
))

In [ ]:
# ── PASO 8: Comparación de 4 Métodos de Imputación ───────────────────────────
print('PASO 8 — Comparación de 4 Métodos de Imputación')

# Usar pct_logro_satisfactorio como variable de análisis principal
cols_num_validas = [c for c in cols_num
                    if master[c].isnull().sum() > 0
                    and master[c].isnull().sum() < len(master)*0.6
                    and master[c].dtype in ['float64', 'int64']]

X_impute = master[cols_num_validas].copy()

metodos = {
    'Media':    SimpleImputer(strategy='mean'),
    'Mediana':  SimpleImputer(strategy='median'),
    'KNN(k=5)': KNNImputer(n_neighbors=5),
    'MICE':     IterativeImputer(max_iter=10, random_state=42),
}

resultados_imp = {}
var_eval = cols_num_validas[0]  # Evaluar sobre la primera variable con nulos
orig_vals = X_impute[var_eval].dropna().values

for nombre, imputer in metodos.items():
    X_imp = pd.DataFrame(imputer.fit_transform(X_impute), columns=cols_num_validas)
    imp_vals = X_imp[var_eval].values
    resultados_imp[nombre] = {
        'Media_Original': orig_vals.mean(),
        'Media_Imputado': imp_vals.mean(),
        'Std_Original':   orig_vals.std(),
        'Std_Imputado':   imp_vals.std(),
        'Delta_Media':    abs(imp_vals.mean() - orig_vals.mean()),
        'Delta_Std':      abs(imp_vals.std()  - orig_vals.std()),
        'X_imp':          X_imp,
    }

df_resultados = pd.DataFrame([{k:v for k,v in r.items() if k!='X_imp'}
                               for r in resultados_imp.values()],
                              index=resultados_imp.keys()).round(4)
df_resultados['Score_Combinado'] = df_resultados['Delta_Media'] + df_resultados['Delta_Std']
mejor_metodo = df_resultados['Score_Combinado'].idxmin()
print(f'\n  🏆 Mejor método: {mejor_metodo} (menor distancia de media+std respecto al original)')
display(df_resultados.style.highlight_min(subset=['Score_Combinado'], color='#D8F3DC'))

In [ ]:
# Visualización comparativa de 4 paneles
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'Paso 8 — Comparación de Métodos de Imputación\nVariable: {var_eval}',
             fontsize=14, fontweight='bold', color='#1B4332')

colores = [VERDE, VERDE_CLARO, AMARILLO, ROJO]

# Panel 1: distribuciones
axes[0,0].hist(orig_vals, bins=30, alpha=0.5, color=GRIS, label='Original (sin nulos)', density=True)
for (nombre, r), color in zip(resultados_imp.items(), colores):
    axes[0,0].hist(r['X_imp'][var_eval].values, bins=30, alpha=0.35, color=color, label=nombre, density=True)
axes[0,0].set_title('Distribuciones superpuestas'); axes[0,0].legend(fontsize=8)

# Panel 2: boxplot comparativo
data_box = [orig_vals] + [r['X_imp'][var_eval].values for r in resultados_imp.values()]
labels_box = ['Original'] + list(resultados_imp.keys())
bp = axes[0,1].boxplot(data_box, labels=labels_box, patch_artist=True)
for patch, color in zip(bp['boxes'], [GRIS]+colores):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0,1].set_title('Boxplot Comparativo')

# Panel 3: barras de preservación de media
nombres_met = list(resultados_imp.keys())
deltas_media = [resultados_imp[m]['Delta_Media'] for m in nombres_met]
bars = axes[1,0].bar(nombres_met, deltas_media, color=colores, alpha=0.8, edgecolor='white')
axes[1,0].set_title('Δ Media (menor = mejor preservación)'); axes[1,0].set_ylabel('|Media_imp - Media_orig|')
axes[1,0].bar_label(bars, fmt='%.3f', padding=3)

# Panel 4: barras de preservación de std
deltas_std = [resultados_imp[m]['Delta_Std'] for m in nombres_met]
bars2 = axes[1,1].bar(nombres_met, deltas_std, color=colores, alpha=0.8, edgecolor='white')
axes[1,1].set_title('Δ Std (menor = mejor preservación)'); axes[1,1].set_ylabel('|Std_imp - Std_orig|')
axes[1,1].bar_label(bars2, fmt='%.3f', padding=3)

plt.tight_layout()
plt.savefig('paso8_comparacion_imputacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── PASO 9: Imputación Final y Validación ────────────────────────────────────
print('PASO 9 — Imputación Final y Validación')

# Aplicar el mejor método
X_final = resultados_imp[mejor_metodo]['X_imp'].copy()

# Para D4: imputar con mediana departamental
for col_d4 in ['pct_pobreza_monetaria', 'idh', 'pct_hogares_sin_internet']:
    if col_d4 in master.columns:
        mediana_dept = master.groupby('dre')[col_d4].transform('median')
        master[col_d4] = master[col_d4].fillna(mediana_dept)

# Actualizar la tabla maestra con los valores imputados
for col in cols_num_validas:
    master[col] = X_final[col].values

# Validación
nulos_restantes = master[cols_num_validas].isnull().sum().sum()
print(f'  Nulos restantes en variables numéricas: {nulos_restantes}')

# Histogramas antes/después para la variable principal
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(orig_vals, bins=30, color=GRIS, alpha=0.8, edgecolor='white')
axes[0].set_title(f'Antes de imputar: {var_eval}\n(solo valores presentes)', color='#1B4332')
axes[1].hist(master[var_eval].dropna(), bins=30, color=VERDE, alpha=0.8, edgecolor='white')
axes[1].set_title(f'Después de imputar ({mejor_metodo})\n(todos los registros)', color='#1B4332')
plt.suptitle('Paso 9 — Validación Visual de Imputación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('paso9_validacion_imputacion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## PARTE 5 — Detección y Tratamiento de Outliers

In [ ]:
var_outlier = var_eval
datos = master[var_outlier].dropna()

# Método 1: IQR
Q1, Q3 = datos.quantile(0.25), datos.quantile(0.75)
IQR = Q3 - Q1
outliers_iqr = master[(master[var_outlier] < Q1 - 1.5*IQR) | (master[var_outlier] > Q3 + 1.5*IQR)]

# Método 2: Z-score
z_scores = np.abs(stats.zscore(datos))
outliers_z = datos[z_scores > 3]

# Método 3: Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
pred_iso = iso.fit_predict(master[[var_outlier]].dropna())
n_outliers_iso = (pred_iso == -1).sum()

print(f'Outliers detectados en {var_outlier}:')
print(f'  IQR:              {len(outliers_iqr):>5} ({len(outliers_iqr)/len(master)*100:.1f}%)')
print(f'  Z-score (|z|>3):  {len(outliers_z):>5} ({len(outliers_z)/len(datos)*100:.1f}%)')
print(f'  Isolation Forest: {n_outliers_iso:>5} ({n_outliers_iso/len(pred_iso)*100:.1f}%)')

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, titulo, serie_outliers in zip(
    axes,
    ['IQR', 'Z-score (|z|>3)', 'Isolation Forest'],
    [outliers_iqr[var_outlier], outliers_z, datos[pred_iso==-1]]
):
    ax.scatter(range(len(datos)), datos, s=5, color=VERDE, alpha=0.4, label='Normal')
    if len(serie_outliers) > 0:
        idx = datos.index.isin(serie_outliers.index)
        ax.scatter(np.where(idx)[0], datos[idx], s=40, color=ROJO, zorder=5, label='Outlier')
    ax.set_title(f'{titulo}: {len(serie_outliers)} outliers', color='#1B4332')
    ax.legend(fontsize=8)
plt.suptitle(f'Parte 5 — Comparación de Métodos de Detección de Outliers\nVariable: {var_outlier}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('parte5_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

# Decisión de tratamiento: winsorización P1-P99
p1 = master[var_outlier].quantile(0.01)
p99 = master[var_outlier].quantile(0.99)
master[var_outlier + '_wins'] = master[var_outlier].clip(p1, p99)
print(f'\n  ✅ Winsorización P1-P99 aplicada. Rango resultante: [{p1:.2f}, {p99:.2f}]')
print('  Justificación: las IE unidocentes con 0% o 100% son reales pero estadísticamente inestables')

---
## PARTE 6 — Transformaciones

In [ ]:
# One-Hot Encoding para variables categóricas
master['area_flag'] = (master['area_geografica'] == 'Rural').astype(int)   # RuralFlag
master['gestion_publica'] = (master['gestion_tipo'] == 'Publica').astype(int)
master['internet_si'] = (master['conectividad_internet'] == 'SI').astype(int)

# One-Hot para 'forma' (agrupando < 2% en 'Otra')
if 'forma' in master.columns:
    freq_forma = master['forma'].value_counts(normalize=True)
    formas_validas = freq_forma[freq_forma >= 0.02].index.tolist()
    master['forma_clean'] = master['forma'].where(master['forma'].isin(formas_validas), other='Otra')
    dummies_forma = pd.get_dummies(master['forma_clean'], prefix='forma', drop_first=True)
    master = pd.concat([master, dummies_forma], axis=1)
    print(f'  Columnas One-Hot de Forma: {dummies_forma.columns.tolist()}')

# Log Transform para IDH y variables con cola derecha
if 'idh' in master.columns and master['idh'].min() > 0:
    master['idh_log'] = np.log1p(master['idh'])

# RobustScaler para % Logro (outliers legítimos)
scaler_robust = RobustScaler()
master['logro_scaled'] = scaler_robust.fit_transform(
    master[[var_outlier + '_wins']].fillna(master[var_outlier + '_wins'].median())
)

print('✅ Transformaciones aplicadas correctamente.')
print(f'   Dimensiones tabla maestra: {master.shape}')

---
## PARTE 7 — Feature Engineering: EQI y SVI

In [ ]:
scaler_mm = MinMaxScaler()

def safe_scale(series):
    """MinMaxScaler robusto a NaN — rellena con mediana antes de escalar."""
    arr = series.fillna(series.median()).values.reshape(-1,1)
    return scaler_mm.fit_transform(arr).flatten()

# ── EQI — Education Quality Index ─────────────────────────────────────────────
logro_col = var_outlier + '_wins'

# Si tenemos varias métricas ECE, promediar; si no, usar la principal
cols_ece = [c for c in master.columns if 'logro' in c.lower() and '_wins' in c]
if len(cols_ece) >= 1:
    master['logro_prom'] = master[cols_ece].mean(axis=1)
else:
    master['logro_prom'] = master[logro_col] if logro_col in master.columns else 50

master['logro_norm']  = safe_scale(master['logro_prom'])

# Tasa de aprobación (proxy: 1 - % en inicio/previo inicio)
if 'pct_logro_inicio' in master.columns:
    master['tasa_aprobacion'] = 100 - master['pct_logro_inicio'].fillna(master['pct_logro_inicio'].median())
    master['aprobacion_norm'] = safe_scale(master['tasa_aprobacion'])
else:
    master['aprobacion_norm'] = master['logro_norm']

# Clima escolar: simulado como combinación de conectividad + logro (proxy simplificado)
master['clima_norm'] = safe_scale(master['logro_norm'] * 0.7 + master['internet_si'] * 0.3)

master['EQI'] = (0.4 * master['logro_norm'] +
                 0.4 * master['clima_norm'] +
                 0.2 * master['aprobacion_norm'])

# ── SVI — Socioeconomic Vulnerability Index ────────────────────────────────────
if 'pct_pobreza_monetaria' in master.columns:
    master['pobreza_norm']   = safe_scale(master['pct_pobreza_monetaria'])
else:
    master['pobreza_norm']   = 0.5

if 'pct_hogares_sin_internet' in master.columns:
    master['sin_internet_norm'] = safe_scale(master['pct_hogares_sin_internet'])
else:
    master['sin_internet_norm'] = 0.5

if 'idh' in master.columns:
    master['idh_inv_norm']   = 1 - safe_scale(master['idh'])
else:
    master['idh_inv_norm']   = 0.5

master['SVI'] = (0.35 * master['pobreza_norm'] +
                 0.35 * master['sin_internet_norm'] +
                 0.30 * master['idh_inv_norm'])

# ── Variables adicionales ───────────────────────────────────────────────────────
if 'n_estudiantes' in master.columns:
    master['StudentDensityCategory'] = pd.cut(
        master['n_estudiantes'].fillna(0),
        bins=[0, 50, 200, np.inf],
        labels=['Bajo', 'Medio', 'Alto'],
        right=False
    )

master['RuralFlag'] = master['area_flag'] if 'area_flag' in master.columns else 0
master['low_quality'] = (master['EQI'] <= master['EQI'].quantile(0.25)).astype(int)

print(f'✅ EQI y SVI calculados.')
print(f'   EQI — Media: {master["EQI"].mean():.3f} | Std: {master["EQI"].std():.3f}')
print(f'   SVI — Media: {master["SVI"].mean():.3f} | Std: {master["SVI"].std():.3f}')
print(f'   IE clasificadas como low_quality: {master["low_quality"].sum()} ({master["low_quality"].mean()*100:.1f}%)')

---
## PARTE 8 — Análisis Exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Parte 8 — Análisis Exploratorio: EQI y SVI', fontsize=15, fontweight='bold', color='#1B4332')

# 1) Distribución del EQI
axes[0,0].hist(master['EQI'].dropna(), bins=40, color=VERDE, alpha=0.8, edgecolor='white')
axes[0,0].axvline(master['EQI'].quantile(0.25), color=ROJO, linestyle='--', lw=2, label='Q25 (umbral low_quality)')
axes[0,0].set_title('Distribución del EQI', fontsize=12, color='#1B4332')
axes[0,0].set_xlabel('Education Quality Index'); axes[0,0].legend()

# 2) Scatter SVI vs EQI
sc = axes[0,1].scatter(master['SVI'], master['EQI'],
                       c=master['low_quality'], cmap='RdYlGn_r',
                       alpha=0.4, s=15)
plt.colorbar(sc, ax=axes[0,1], label='low_quality')
m, b = np.polyfit(master['SVI'].fillna(0), master['EQI'].fillna(0), 1)
x_line = np.linspace(0, 1, 100)
axes[0,1].plot(x_line, m*x_line+b, 'k--', lw=1.5, label=f'Tendencia (m={m:.2f})')
axes[0,1].set_title('SVI vs EQI — ¿El contexto adverso predice baja calidad?', fontsize=11, color='#1B4332')
axes[0,1].set_xlabel('SVI'); axes[0,1].set_ylabel('EQI'); axes[0,1].legend()

# 3) Boxplot EQI por Gestión
if 'gestion_tipo' in master.columns:
    grupos_gest = [master[master['gestion_tipo']==g]['EQI'].dropna()
                   for g in master['gestion_tipo'].dropna().unique()]
    labels_gest = master['gestion_tipo'].dropna().unique().tolist()
    bp = axes[1,0].boxplot(grupos_gest, labels=labels_gest, patch_artist=True)
    for patch, c in zip(bp['boxes'], [VERDE, AMARILLO]):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    axes[1,0].set_title('EQI por Tipo de Gestión', fontsize=12, color='#1B4332')
    axes[1,0].set_ylabel('EQI')

# 4) Mapa de calor de correlaciones
cols_corr = ['EQI', 'SVI', 'pct_pobreza_monetaria', 'idh', 'pct_hogares_sin_internet',
             'RuralFlag', 'internet_si', 'n_docentes', 'n_estudiantes']
cols_corr_validas = [c for c in cols_corr if c in master.columns and master[c].dtype in ['float64','int64','int32']]
corr_mat = master[cols_corr_validas].corr()
mask_corr = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask_corr, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=axes[1,1],
            annot_kws={'size': 8})
axes[1,1].set_title('Mapa de Correlaciones', fontsize=12, color='#1B4332')

plt.tight_layout()
plt.savefig('parte8_eda.png', dpi=150, bbox_inches='tight')
plt.show()

---
## PARTE 9 — Modelado Analítico
### Modelo 1 — XGBoost (Clasificación)

In [ ]:
# ── Preparar features ─────────────────────────────────────────────────────────
feature_cols = (['SVI', 'RuralFlag', 'internet_si', 'gestion_publica'] +
                [c for c in master.columns if 'forma_' in c] +
                [c for c in ['n_docentes', 'n_estudiantes', 'pct_pobreza_monetaria', 'idh']
                 if c in master.columns])

feature_cols = [c for c in feature_cols if c in master.columns and master[c].dtype in ['float64','int64','int32']]

target = 'low_quality'
df_model = master[feature_cols + [target]].dropna()
X = df_model[feature_cols]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'  Train: {X_train.shape} | Test: {X_test.shape}')
print(f'  Desbalance: {y_train.value_counts(normalize=True).round(3).to_dict()}')

# ── Entrenar XGBoost ──────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    random_state=42, eval_metric='logloss', verbosity=0
)
xgb.fit(X_train, y_train)

y_prob = xgb.predict_proba(X_test)[:,1]
y_pred = xgb.predict(X_test)
auc = roc_auc_score(y_test, y_prob)
print(f'\n  🎯 XGBoost AUC-ROC: {auc:.4f}')
print('\n', classification_report(y_test, y_pred, target_names=['Alta calidad','Baja calidad']))

In [ ]:
# Visualizar Feature Importance + Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Modelo 1 — XGBoost: Evaluación y Feature Importance',
             fontsize=14, fontweight='bold', color='#1B4332')

importances = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=True)
colores_fi = [ROJO if importances.iloc[i] == importances.max() else VERDE
              for i in range(len(importances))]
importances.plot.barh(ax=axes[0], color=colores_fi[-len(importances):], edgecolor='white')
axes[0].set_title('Feature Importance (Gain)', color='#1B4332')
axes[0].set_xlabel('Importancia relativa')

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Alta calidad', 'Baja calidad'])
disp.plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title(f'Matriz de Confusión (AUC={auc:.3f})', color='#1B4332')

plt.tight_layout()
plt.savefig('parte9_xgboost.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n  🏆 Variable más importante: {importances.idxmax()}')
print('  ⚠️  Correlación ≠ causalidad: la importancia indica capacidad predictiva, no efecto causal.')

### Modelo 2 — K-Means Clustering

In [ ]:
X_cluster = master[['EQI', 'SVI']].dropna()

# Determinar K óptimo con Silhouette Score
silhouettes = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    sil = silhouette_score(X_cluster, labels)
    silhouettes[k] = sil

K_optimo = max(silhouettes, key=silhouettes.get)
print(f'Silhouette Scores por K: {silhouettes}')
print(f'\n  ✅ K óptimo: {K_optimo} (Silhouette = {silhouettes[K_optimo]:.3f})')

# Entrenar con K óptimo
km_final = KMeans(n_clusters=K_optimo, random_state=42, n_init=10)
master['cluster'] = np.nan
master.loc[X_cluster.index, 'cluster'] = km_final.fit_predict(X_cluster)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Modelo 2 — K-Means Clustering (K={K_optimo})',
             fontsize=14, fontweight='bold', color='#1B4332')

# Silhouette curve
axes[0].plot(list(silhouettes.keys()), list(silhouettes.values()),
             'o-', color=VERDE, lw=2, markersize=8)
axes[0].axvline(K_optimo, color=ROJO, linestyle='--', label=f'K óptimo = {K_optimo}')
axes[0].set_title('Silhouette Score por número de clusters', color='#1B4332')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Silhouette Score'); axes[0].legend()

# Clusters en espacio EQI-SVI
colores_cluster = [VERDE, AMARILLO, ROJO, VERDE_CLARO, GRIS]
for i in range(K_optimo):
    mask = master['cluster'] == i
    axes[1].scatter(master.loc[mask, 'SVI'], master.loc[mask, 'EQI'],
                    s=15, alpha=0.5, color=colores_cluster[i], label=f'Cluster {i+1}')

# Centroides
centroids = km_final.cluster_centers_
axes[1].scatter(centroids[:,1], centroids[:,0], s=200, marker='*',
                color='black', zorder=10, label='Centroides')
axes[1].set_title('Segmentación IE en espacio EQI-SVI', color='#1B4332')
axes[1].set_xlabel('SVI (Vulnerabilidad Socioeconómica)')
axes[1].set_ylabel('EQI (Calidad Educativa)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('parte9_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Perfil de cada cluster
print('═══ Perfil de Clusters ═══')
cols_perfil = ['EQI', 'SVI', 'pct_pobreza_monetaria', 'idh',
               'RuralFlag', 'internet_si', 'n_docentes']
cols_perfil_validas = [c for c in cols_perfil if c in master.columns]

perfil_clusters = master.groupby('cluster')[cols_perfil_validas].mean().round(3)
perfil_clusters['N_IE'] = master.groupby('cluster').size()
perfil_clusters['Pct_low_quality'] = master.groupby('cluster')['low_quality'].mean().round(3)
display(perfil_clusters.style.background_gradient(subset=['EQI'], cmap='Greens')
                             .background_gradient(subset=['SVI'], cmap='Reds_r'))

print('\n  💡 Interpretación sugerida de clusters:')
print('  Cluster con EQI alto + SVI bajo  → IE de alta calidad en contexto favorable')
print('  Cluster con EQI bajo  + SVI alto → IE de baja calidad en contexto vulnerable → PRIORIDAD DE INTERVENCIÓN')
print('  Cluster con EQI bajo  + SVI bajo → IE con desempeño bajo sin justificación contextual clara')

### Modelo 3 — Random Forest (Validación cruzada)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42,
                             class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_rf = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')

print(f'Random Forest — AUC-ROC (5-fold CV):')
print(f'  Media: {auc_rf.mean():.4f} | Std: {auc_rf.std():.4f}')
print(f'  Por fold: {auc_rf.round(4)}')
print(f'\n  XGBoost (holdout): {auc:.4f}')

# Feature importance comparada
rf.fit(X_train, y_train)
fi_rf = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
fi_xgb = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)

print('\n  Top 5 variables más importantes:')
comparacion = pd.DataFrame({'Random Forest': fi_rf.head(5).round(4),
                             'XGBoost': fi_xgb.reindex(fi_rf.head(5).index).round(4)})
display(comparacion)

if fi_rf.idxmax() == fi_xgb.idxmax():
    print(f'\n  ✅ CONCLUSIÓN ROBUSTA: ambos modelos coinciden en que "{fi_rf.idxmax()}" es la variable más importante.')
else:
    print(f'\n  ⚠️ Los modelos difieren: RF={fi_rf.idxmax()} vs XGBoost={fi_xgb.idxmax()}. Analizar más profundamente.')

---
## PARTE 10 — Interpretación para el Negocio

> **Objetivo:** Traducir los hallazgos técnicos en conclusiones accionables para el Director Regional de Educación (DRE).

### Resumen Ejecutivo (borrador)

In [ ]:
n_ie_total = len(master)
n_low = master['low_quality'].sum()
pct_low = n_low / n_ie_total * 100
top_variable = fi_rf.idxmax()
n_clusters = K_optimo
corr_svi_eqi = master['SVI'].corr(master['EQI'])

resumen = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║           RESUMEN EJECUTIVO — CALIDAD EDUCATIVA EN PERÚ                    ║
╚══════════════════════════════════════════════════════════════════════════════╝

HALLAZGO 1 — MAGNITUD DEL PROBLEMA
  De las {n_ie_total:,} instituciones educativas analizadas, el {pct_low:.1f}% ({n_low:,} IE)
  se encuentran en el cuartil inferior de calidad (EQI ≤ Q25). Estas IE son
  candidatas prioritarias para intervención inmediata.

HALLAZGO 2 — FACTOR PREDICTOR PRINCIPAL
  Ambos modelos (XGBoost AUC={auc:.3f} y Random Forest AUC={auc_rf.mean():.3f})
  coinciden en que '{top_variable}' es el factor con mayor capacidad
  predictiva de baja calidad educativa.

HALLAZGO 3 — CONTEXTO SOCIOECONÓMICO
  La correlación entre SVI y EQI es de {corr_svi_eqi:.3f}, lo que sugiere que
  {'una relación negativa moderada-fuerte: mayor vulnerabilidad socioeconómica se asocia con menor calidad educativa.' if corr_svi_eqi < -0.2 else 'una relación débil: el contexto socioeconómico no explica completamente las diferencias de calidad.'}

HALLAZGO 4 — SEGMENTACIÓN ({n_clusters} CLUSTERS)
  El análisis de clustering identificó {n_clusters} segmentos diferenciados de IE.
  Se recomienda aplicar estrategias diferenciadas por segmento (ver tabla de perfil).

RECOMENDACIONES POR SEGMENTO:
  • IE alta vulnerabilidad + bajo EQI  → Intervención urgente: refuerzo docente,
    conectividad, materiales pedagógicos, acompañamiento familiar.
  • IE baja vulnerabilidad + bajo EQI  → Auditoría de gestión institucional;
    posible problema de liderazgo directivo o prácticas pedagógicas.
  • IE alta vulnerabilidad + alto EQI  → Documentar y escalar buenas prácticas;
    estos son los 'campeones contra el contexto adverso'.

LIMITACIONES DEL ANÁLISIS:
  • Datos socioeconómicos con hasta 2 años de desfase temporal.
  • IE de modalidad alternativa y EIB excluidas del modelo principal.
  • Correlación estadística ≠ causalidad. Los modelos identifican patrones,
    no relaciones causales verificadas.
"""
print(resumen)

---
## 💾 Exportar Resultados

In [ ]:
# Exportar tabla maestra con EQI, SVI, cluster y predicciones
cols_export = (['cod_mod', 'dre', 'distrito', 'area_geografica', 'gestion_tipo',
                'EQI', 'SVI', 'cluster', 'low_quality', 'RuralFlag'] +
               [c for c in ['pct_pobreza_monetaria', 'idh', 'n_docentes', 'n_estudiantes']
                if c in master.columns])
cols_export_validas = [c for c in cols_export if c in master.columns]

tabla_export = master[cols_export_validas].copy()
tabla_export['prob_baja_calidad'] = np.nan
tabla_export.loc[df_model.index, 'prob_baja_calidad'] = (
    xgb.predict_proba(df_model[feature_cols])[:,1].round(4)
)

tabla_export.to_csv('tabla_maestra_educacion_peru.csv', index=False, encoding='utf-8-sig')
print(f'✅ Tabla maestra exportada: {tabla_export.shape}')
print('   Archivo: tabla_maestra_educacion_peru.csv')

# Descargar desde Colab
try:
    from google.colab import files
    files.download('tabla_maestra_educacion_peru.csv')
    print('   ⬇️ Descarga iniciada automáticamente.')
except:
    print('   (Para descargar en Colab: ejecuta from google.colab import files; files.download("tabla_maestra_educacion_peru.csv"))')

print('\n🎓 Análisis completo. Revisa las preguntas de reflexión en el PDF del caso de estudio.')